# LLM v3 Preset Ablation Comparison

This notebook loads the named validation-set ablation runs and compares them on:

- mean 2Q gate reduction (%)
- mean depth reduction (%)
- mean runtime (seconds)

It is resilient to missing or incomplete runs: it shows a status table first, then loads only the completed runs.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "machine_learning" / "LLM" / "llm_v3").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing machine_learning/LLM/llm_v3.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from machine_learning.LLM.llm_v3.analysis_utils import load_manifest, load_predictions, load_summary

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"Repo root: {REPO_ROOT}")


In [ ]:
ARTIFACT_ROOT = REPO_ROOT / "machine_learning" / "LLM" / "llm_v3" / "artifacts"
SPLIT = "validation"
REQUIRE_ALL_RUNS = False

RUN_SPECS = [
    ("clean_all_candidates", "clean_all_candidates_validation"),
    ("profiles_only_all_candidates", "profiles_only_all_candidates_validation"),
    ("retrieval_only_all_candidates", "retrieval_only_all_candidates_validation"),
    ("gate_only_all_candidates", "gate_only_all_candidates_validation"),
    ("shortlist_only_top10", "shortlist_only_top10_validation"),
    ("retrieval_plus_gate_all_candidates", "retrieval_plus_gate_all_candidates_validation"),
    ("all_prompt_extras_all_candidates", "all_prompt_extras_all_candidates_validation"),
    ("full_system", "full_system_validation"),
]

PLOT_ORDER = [preset for preset, _ in RUN_SPECS]


def expected_run_dir(directory_name: str) -> Path:
    return ARTIFACT_ROOT / directory_name


def run_status(run_dir: Path) -> dict:
    manifest_path = run_dir / "manifest.json"
    predictions_path = run_dir / "predictions.csv"
    summary_path = run_dir / "summary.json"
    exists = run_dir.exists()
    manifest_exists = manifest_path.exists()
    predictions_exists = predictions_path.exists()
    summary_exists = summary_path.exists()
    complete = manifest_exists and predictions_exists and summary_exists
    status = None
    if manifest_exists:
        try:
            status = load_manifest(run_dir).get("status")
        except Exception:
            status = "manifest_unreadable"
    return {
        "run_dir": str(run_dir),
        "exists": exists,
        "manifest": manifest_exists,
        "predictions": predictions_exists,
        "summary": summary_exists,
        "complete": complete,
        "status": status,
    }


In [ ]:
status_rows = []
for preset_name, directory_name in RUN_SPECS:
    row = {"preset": preset_name}
    row.update(run_status(expected_run_dir(directory_name)))
    status_rows.append(row)

status_df = pd.DataFrame(status_rows)
display(status_df)

missing_or_incomplete = status_df[~status_df["complete"]].copy()
if REQUIRE_ALL_RUNS and not missing_or_incomplete.empty:
    raise FileNotFoundError(
        "These runs are missing or incomplete:\n"
        + missing_or_incomplete[["preset", "run_dir", "status"]].to_string(index=False)
    )

if not missing_or_incomplete.empty:
    print("Skipping missing/incomplete runs:")
    display(missing_or_incomplete[["preset", "status", "run_dir"]])


In [ ]:
completed_specs = [
    (preset_name, directory_name)
    for preset_name, directory_name in RUN_SPECS
    if bool(status_df.loc[status_df["preset"] == preset_name, "complete"].iloc[0])
]

if not completed_specs:
    raise FileNotFoundError("No completed preset validation runs were found.")

runs = {}
for preset_name, directory_name in completed_specs:
    run_dir = expected_run_dir(directory_name)
    runs[preset_name] = {
        "run_dir": run_dir,
        "manifest": load_manifest(run_dir),
        "summary": load_summary(run_dir),
        "predictions": load_predictions(run_dir),
    }

print(f"Loaded {len(runs)} completed {SPLIT} runs.")


In [ ]:
def metric_from_summary_or_predictions(payload: dict, summary_key: str, prediction_key: str) -> float:
    summary_value = payload["summary"].get(summary_key)
    if summary_value is not None:
        return float(summary_value)
    return float(pd.to_numeric(payload["predictions"][prediction_key], errors="coerce").mean())


comparison_rows = []
for preset_name, payload in runs.items():
    manifest = payload["manifest"]
    summary = payload["summary"]
    comparison_rows.append(
        {
            "preset": preset_name,
            "model_name": manifest.get("model_name"),
            "circuits": int(summary.get("num_circuits", len(payload["predictions"]))),
            "mean_2q_reduction_pct": metric_from_summary_or_predictions(
                payload,
                "mean_two_qubit_gate_reduction_pct",
                "two_qubit_gate_reduction_pct",
            ),
            "mean_depth_reduction_pct": metric_from_summary_or_predictions(
                payload,
                "mean_depth_reduction_pct",
                "depth_reduction_pct",
            ),
            "mean_runtime_seconds": metric_from_summary_or_predictions(
                payload,
                "mean_runtime_seconds",
                "runtime_seconds",
            ),
            "mean_objective_score": metric_from_summary_or_predictions(
                payload,
                "mean_objective_score",
                "objective_score",
            ),
            "mean_regret": float(summary.get("mean_regret", pd.to_numeric(payload["predictions"].get("regret"), errors="coerce").mean())),
            "original_selection_rate": float(summary.get("original_selection_rate", pd.to_numeric(payload["predictions"].get("selected_is_original"), errors="coerce").mean())),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
available_order = [preset for preset in PLOT_ORDER if preset in comparison_df["preset"].tolist()]
comparison_df["preset"] = pd.Categorical(comparison_df["preset"], categories=available_order, ordered=True)
comparison_df = comparison_df.sort_values("preset").reset_index(drop=True)
display(comparison_df)


In [ ]:
display_columns = [
    "preset",
    "mean_2q_reduction_pct",
    "mean_depth_reduction_pct",
    "mean_runtime_seconds",
    "mean_objective_score",
    "mean_regret",
    "original_selection_rate",
]

display(
    comparison_df[display_columns].style.format(
        {
            "mean_2q_reduction_pct": "{:.2f}",
            "mean_depth_reduction_pct": "{:.2f}",
            "mean_runtime_seconds": "{:.2f}",
            "mean_objective_score": "{:.4f}",
            "mean_regret": "{:.4f}",
            "original_selection_rate": "{:.3f}",
        }
    )
)


In [ ]:
plot_df = comparison_df.copy()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.barplot(data=plot_df, x="mean_2q_reduction_pct", y="preset", ax=axes[0], color="#5B7DB1")
axes[0].set_title("Mean 2Q Gate Reduction (%)")
axes[0].set_xlabel("Percent")
axes[0].set_ylabel("")

sns.barplot(data=plot_df, x="mean_depth_reduction_pct", y="preset", ax=axes[1], color="#7A93C8")
axes[1].set_title("Mean Depth Reduction (%)")
axes[1].set_xlabel("Percent")
axes[1].set_ylabel("")

runtime_plot_df = plot_df.sort_values("mean_runtime_seconds", ascending=True).reset_index(drop=True)
sns.barplot(data=runtime_plot_df, x="mean_runtime_seconds", y="preset", ax=axes[2], color="#6FAFA0")
axes[2].set_title("Mean Runtime (Seconds)")
axes[2].set_xlabel("Seconds")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
baseline_name = "clean_all_candidates"
if baseline_name not in runs:
    raise KeyError(
        "The clean_all_candidates baseline is required for the delta comparison, "
        "but its run is not completed yet."
    )

baseline_predictions = runs[baseline_name]["predictions"][[
    "circuit_id",
    "two_qubit_gate_reduction_pct",
    "depth_reduction_pct",
    "runtime_seconds",
]].rename(
    columns={
        "two_qubit_gate_reduction_pct": "baseline_2q",
        "depth_reduction_pct": "baseline_depth",
        "runtime_seconds": "baseline_runtime",
    }
)

delta_rows = []
for preset_name, payload in runs.items():
    if preset_name == baseline_name:
        continue
    merged = payload["predictions"][[
        "circuit_id",
        "two_qubit_gate_reduction_pct",
        "depth_reduction_pct",
        "runtime_seconds",
    ]].merge(baseline_predictions, on="circuit_id", how="inner")
    delta_rows.append(
        {
            "preset": preset_name,
            "delta_2q_vs_clean": float((merged["two_qubit_gate_reduction_pct"] - merged["baseline_2q"]).mean()),
            "delta_depth_vs_clean": float((merged["depth_reduction_pct"] - merged["baseline_depth"]).mean()),
            "delta_runtime_vs_clean": float((merged["runtime_seconds"] - merged["baseline_runtime"]).mean()),
        }
    )

delta_df = pd.DataFrame(delta_rows)
if not delta_df.empty:
    delta_df["preset"] = pd.Categorical(delta_df["preset"], categories=[preset for preset in PLOT_ORDER if preset != baseline_name and preset in delta_df["preset"].tolist()], ordered=True)
    delta_df = delta_df.sort_values("preset").reset_index(drop=True)

display(
    delta_df.style.format(
        {
            "delta_2q_vs_clean": "{:+.2f}",
            "delta_depth_vs_clean": "{:+.2f}",
            "delta_runtime_vs_clean": "{:+.2f}",
        }
    )
)


## Notes

- Higher `mean_2q_reduction_pct` is better.
- Higher `mean_depth_reduction_pct` is better.
- Lower `mean_runtime_seconds` is better.
- In the delta table, positive runtime deltas mean the preset is slower than `clean_all_candidates`.
- Set `REQUIRE_ALL_RUNS = True` near the top if you want the notebook to fail until all eight runs are complete.
